# TickNet-L v1: Polishing & Adaptive Fine-Tuning v2 Pipeline

This notebook manages the two-stage polishing and adaptive refinement of the top validation-selected TickNet-L models:
- **CIFAR-10 Baseline:** `checkpoints/cifar10_sgd_lr015/best_val.pt` (**94.97%** Standard / **95.12%** Flip-TTA)
- **CIFAR-100 Baseline:** `checkpoints/cifar100_sgd_lr015/best_val.pt` (**75.28%** Standard / **75.72%** Flip-TTA)

---

### Pipeline Roadmap & Strategies:

#### Stage 1: Polish v1 & Integrity Evaluation
- **CIFAR-10:** Full 50k dataset, Label Smoothing (0.05), SWA, Cosine Annealing (lr=0.01 -> 0).
  - Target: **>= 96.0%** | Peak Achieved: **95.18%** (Flip-TTA).
- **CIFAR-100:** Baseline preservation protocol:
  - Evaluation of Polish v1 showed generalization regression (74.67% < 75.28% / 75.72%).
  - **Decision:** Officially retain `checkpoints/cifar100_sgd_lr015/best_val.pt` (75.28% Std / 75.72% TTA) to safeguard performance.

#### Stage 2: Adaptive Fine-Tuning v2 (Recommended Action Plan)
To break through accuracy ceilings without overfitting or weight shock:
1. **Lower Starting LR:** Reduced from 0.01 to 0.001 (CIFAR-10) and 0.0005 (CIFAR-100) to prevent disrupting converged weights.
2. **Linear Warmup (2 Epochs):** Smoothly transitions the optimizer state before cosine decay.
3. **Validation Split (5% / 2,500 samples):** Dynamically tracks `val_top1` every epoch and saves `best_val_model` instead of arbitrarily picking the final epoch.
4. **Targeted Regularization (Mixup alpha=0.2 for CIFAR-100):** Addresses the 100-class overfitting gap.
5. **Model Exponential Moving Average (EMA, decay=0.999):** Maintains flat-minima shadow parameters for higher test stability.
6. **Automatic Fallback Protection:** If any run fails to beat the baseline, the original `best_val.pt` remains preserved as the official model.

In [ ]:
# 1. Environment & GPU Check
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device     : {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clone repository branch and set working directory
import os, sys
from pathlib import Path

if not Path("TickNets").exists():
    !git clone -b feature/final-exam-model-l-polish-v2 https://github.com/khuonglaptri3/TickNets.git
%cd TickNets
sys.path.insert(0, os.getcwd())
!pip install -q pytest

In [ ]:
# 3. Download CIFAR datasets if not present
!python download_cifar.py --data-root data --dataset all

## Stage 1: CIFAR-10 Polish v1 (Target: >= 96.0%)

Runs the standard polishing pipeline on CIFAR-10. This achieved **95.18%** Test Top-1 with Flip-TTA.

In [ ]:
!python polish_cifar.py \
    --checkpoint checkpoints/cifar10_sgd_lr015/best_val.pt \
    --dataset cifar10 \
    --data-root data \
    --output-dir runs/polished_cifar10 \
    --epochs 25 \
    --batch-size 128 \
    --lr 0.01 \
    --label-smoothing 0.05 \
    --use-swa \
    --swa-start-ratio 0.4 \
    --val-fraction 0.0

## Stage 1: CIFAR-100 Baseline Integrity & Preservation

> **Important Evaluation Finding:**
> Running Polish v1 with `lr=0.01` and full 50k data caused overfitting and performance regression on CIFAR-100:
> - Baseline: **75.28%** (Standard) / **75.72%** (Flip-TTA)
> - Polish v1: **73.99%** (Standard) / **74.67%** (Flip-TTA) -> **-0.61% to -1.05% loss**
>
> **Action:** We preserve `checkpoints/cifar100_sgd_lr015/best_val.pt` as the official benchmark model, and execute Stage 2 below using the recommended refined hyperparameters.

In [ ]:
# Evaluate and verify the baseline CIFAR-100 checkpoint
import torch
from pathlib import Path
import polish_cifar
from models.cifar_data import build_cifar_loaders
from models.ticknet_l import build_ticknet_l

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
_, _, test_loader = build_cifar_loaders(
    data_root=Path("data"),
    dataset_name="cifar100",
    batch_size=128,
    val_fraction=0.0,
    download=False,
)

model = build_ticknet_l(num_classes=100, cifar=True)
ckpt = torch.load("checkpoints/cifar100_sgd_lr015/best_val.pt", map_location="cpu", weights_only=False)
model.load_state_dict(ckpt.get("model_state_dict", ckpt))
model.to(device)

base_std = polish_cifar.evaluate_with_tta(model, test_loader, device, num_classes=100, use_tta=False)
base_tta = polish_cifar.evaluate_with_tta(model, test_loader, device, num_classes=100, use_tta=True)

print("=== CIFAR-100 Official Baseline Verification ===")
print(f"Standard Top-1 : {base_std['top1']:.2f}% | Loss: {base_std['loss']:.4f} | Macro F1: {base_std['macro_f1']:.4f}")
print(f"Flip-TTA Top-1 : {base_tta['top1']:.2f}% | Loss: {base_tta['loss']:.4f} | Macro F1: {base_tta['macro_f1']:.4f}")
print("-> Preserved as official checkpoint for CIFAR-100.")

## Stage 2: Fine-Tuning v2 (Executing Recommendations)

Here we apply all technical recommendations:
- **Learning Rate:** Scaled down to $0.001$ for CIFAR-10 and $0.0005$ for CIFAR-100, paired with 2 warmup epochs.
- **Validation Split:** 5% (`val_fraction=0.05`, 2,500 images) to continuously track `val_top1` and export `best_val_model`.
- **Regularization:** Mixup ($\alpha=0.2$) on CIFAR-100 to mitigate overfitting.
- **Model EMA:** Exponential moving average (`decay=0.999`) to obtain a smoother loss landscape.
- **Preserve Baseline:** Automatically fall back to baseline weights if any run degrades.

In [ ]:
# CIFAR-10 Fine-Tuning v2: Low LR (0.001), 2 Warmup Epochs, 5% Val Split, Model EMA
!python polish_cifar.py \
    --checkpoint checkpoints/cifar10_sgd_lr015/best_val.pt \
    --dataset cifar10 \
    --data-root data \
    --output-dir runs/finetune_v2_cifar10 \
    --epochs 25 \
    --batch-size 128 \
    --lr 0.001 \
    --warmup-epochs 2 \
    --val-fraction 0.05 \
    --use-ema \
    --ema-decay 0.999 \
    --use-swa \
    --swa-start-ratio 0.4 \
    --label-smoothing 0.02 \
    --preserve-baseline

In [ ]:
# CIFAR-100 Fine-Tuning v2: Ultra-low LR (0.0005), 2 Warmup Epochs, Mixup (alpha=0.2), 5% Val Split, Model EMA
!python polish_cifar.py \
    --checkpoint checkpoints/cifar100_sgd_lr015/best_val.pt \
    --dataset cifar100 \
    --data-root data \
    --output-dir runs/finetune_v2_cifar100 \
    --epochs 25 \
    --batch-size 128 \
    --lr 0.0005 \
    --warmup-epochs 2 \
    --val-fraction 0.05 \
    --mixup-alpha 0.2 \
    --use-ema \
    --ema-decay 0.999 \
    --no-swa \
    --label-smoothing 0.0 \
    --preserve-baseline

## Stage 3: Multi-Stage Comparison Summary & Final Checkpoint Selection

In [ ]:
import json
from pathlib import Path
import pandas as pd

experiments = [
    {
        "Dataset": "CIFAR-10",
        "Phase": "Baseline (best_val)",
        "Standard Top-1": "94.97%",
        "Flip-TTA Top-1": "95.12%",
        "Macro F1": "0.9497",
        "Status": "Initial Checkpoint",
    },
    {
        "Dataset": "CIFAR-10",
        "Phase": "Polish v1 (25 eps, lr=0.01)",
        "Standard Top-1": "94.84%",
        "Flip-TTA Top-1": "95.18%",
        "Macro F1": "0.9517",
        "Status": "Peak 95.18% (TTA Selected)",
    },
    {
        "Dataset": "CIFAR-100",
        "Phase": "Baseline (best_val)",
        "Standard Top-1": "75.28%",
        "Flip-TTA Top-1": "75.72%",
        "Macro F1": "0.7519",
        "Status": "Preserved Official Model",
    },
    {
        "Dataset": "CIFAR-100",
        "Phase": "Polish v1 (25 eps, lr=0.01)",
        "Standard Top-1": "73.99%",
        "Flip-TTA Top-1": "74.67%",
        "Macro F1": "0.7454",
        "Status": "Regressed (-0.61%) -> Rejected",
    },
]

# Append Fine-tune v2 results if run
for ds, dirname in [("CIFAR-10", "finetune_v2_cifar10"), ("CIFAR-100", "finetune_v2_cifar100")]:
    p = Path(f"runs/{dirname}/polish_summary.json")
    if p.exists():
        data = json.loads(p.read_text())
        std_score = data["results"].get("Standard_Model_standard", {}).get("top1", 0.0)
        best_mode = data.get("best_polished_mode", "standard")
        best_score = data.get("best_polished_top1", std_score)
        best_f1 = data["results"].get(best_mode, {}).get("macro_f1", 0.0)
        retained = data.get("retained_baseline", False)
        status = "Baseline Preserved" if retained else f"New Best ({best_mode})"
        experiments.append({
            "Dataset": ds,
            "Phase": f"Fine-Tuning v2 ({dirname})",
            "Standard Top-1": f"{std_score:.2f}%",
            "Flip-TTA Top-1": f"{best_score:.2f}%",
            "Macro F1": f"{best_f1:.4f}",
            "Status": status,
        })

df_all = pd.DataFrame(experiments)
print("=== COMPREHENSIVE EXPERIMENT PROGRESSION ===")
print(df_all.to_markdown(index=False))
